In [ ]:
!pip install -U fashion_clip
!pip install opencv-python

In [ ]:
# from fashion_clip.fashion_clip import FashionCLIP
import numpy as np
import pandas as pd
import os 
import json
import shutil
import cv2

DATA_PATH = "/workspace/CloSe/data"
EVAL_PATH = "/workspace/data/3d_data"
eval_dict = {
    "close_image_scan": {
        "cg_baseline": [],
        "cg_cot": [],
        "cg_blip": [],
        "cg_retrieval": []
    },
    "s_sketch": {
        "cg_baseline": [],
        "cg_cot": [],
        "cg_blip": [],
    },
    "example_data": {
        "cg_retrieval": [],
        "cg_cot": [],
        "cg_blip": []
    }
}

dataset_dict = {}

In [26]:
def all_sub_images(path):
    files = os.listdir(path)
    sub_images = []
    for file in files:
        if file.endswith(".png"):
            file_path = os.path.join(path, file)
            sub_images.append(file_path)
    return sub_images


def get_all_sub_dirs(path):
    dir_list = os.listdir(path)
    sub_dirs = []
    dir_names = []
    for file in dir_list:
        sub_path = os.path.join(path, file)
        if os.path.isdir(sub_path):
            sub_dirs.append(sub_path)
            dir_names.append(file)
    return sub_dirs, dir_names

def extract_images(folder, folder_name, output_folder, output_path = "/workspace/CloSe/evaluation"):
    output_path = os.path.join(output_path, output_folder)
    os.makedirs(output_path, exist_ok=True)
    sub_dirs, dir_names = get_all_sub_dirs(folder)
    image_path = []
    for sub_dir, dir_name in zip(sub_dirs, dir_names):
        path = os.path.join(sub_dir, dir_name, f"{dir_name}_render_front.png")
        new_path = os.path.join(output_path, f"{folder_name[14:]}_{dir_name}.png")
        image_path.append(path)
    # use cv2 to concatenate the images
    images = [cv2.imread(image) for image in image_path]
    images = np.concatenate(images, axis=1)
    cv2.imwrite(os.path.join(output_path, f"{folder_name[14:]}.png"), images)
    return image_path

DATASET_JSON = "/workspace/CloSe/data/dataset_dict.json"
EVAL_JSON = "/workspace/CloSe/data/eval_dict.json"
if os.path.exists(DATASET_JSON):
    with open(DATASET_JSON, "r") as f:
        dataset_dict = json.load(f)
else:
    dataset_dict = {}

    for dataset in eval_dict.keys():
        dataset_dict[dataset] = all_sub_images(os.path.join(DATA_PATH, dataset))
        print(f"Loaded {len(dataset_dict[dataset])} {dataset} sub directories")

    with open(DATASET_JSON, "w") as f:
        json.dump(dataset_dict, f, indent=4)
        
for dataset, methods in eval_dict.items():
    for method, _ in methods.items():
        path = os.path.join(EVAL_PATH, f"{dataset}_{method}", "vis_new")
        sub_dirs, dir_names = get_all_sub_dirs(path)
        for sub_dir, dir_name in zip(sub_dirs, dir_names):
            eval_dict[dataset][method] += extract_images(sub_dir, dir_name, f"{dataset}_{method}")
with open(EVAL_JSON, "w") as f:
    json.dump(eval_dict, f, indent=4)

In [23]:
import cv2

def resize_image_height(methods, height):
    for method, images in methods.items():
        print(f"Method: {method}")
        for image in images:
            img = cv2.imread(image)
            img = cv2.resize(img, (int(img.shape[1] * height / img.shape[0]), height))
            cv2.imwrite(image, img)

def resize_image_for_dict(data_dict, height):
    for dataset, methods in data_dict.items():
        print(f"Resizing {dataset} images to {height} height")
        resize_image_height(methods, height)

In [ ]:
resize_image_for_dict(eval_dict, 512)

Resizing close_image_scan images to 512 height
Method: cg_baseline


Method: cg_cot
Method: cg_blip
Method: cg_retrieval
Resizing s_sketch images to 512 height
Method: cg_baseline
Method: cg_cot
Method: cg_blip
Resizing example_data images to 512 height
Method: cg_retrieval
Method: cg_cot
Method: cg_blip


In [24]:
resize_image_height(dataset_dict, 512)

Method: close_image_scan


Method: s_sketch
Method: example_data


In [28]:
dataset_dict

{'close_image_scan': ['/workspace/CloSe/data/close_image_scan/10001_1923.png',
  '/workspace/CloSe/data/close_image_scan/10005_2038.png',
  '/workspace/CloSe/data/close_image_scan/10009_2154.png',
  '/workspace/CloSe/data/close_image_scan/10010_2213.png',
  '/workspace/CloSe/data/close_image_scan/10011_2219.png',
  '/workspace/CloSe/data/close_image_scan/10012_2321.png',
  '/workspace/CloSe/data/close_image_scan/10015_2572.png',
  '/workspace/CloSe/data/close_image_scan/10018_4016.png',
  '/workspace/CloSe/data/close_image_scan/10020_2774.png',
  '/workspace/CloSe/data/close_image_scan/10021_2853.png',
  '/workspace/CloSe/data/close_image_scan/10022_2915.png',
  '/workspace/CloSe/data/close_image_scan/10023_2976.png',
  '/workspace/CloSe/data/close_image_scan/10025_3124.png',
  '/workspace/CloSe/data/close_image_scan/10026_3189.png',
  '/workspace/CloSe/data/close_image_scan/10027_3256.png',
  '/workspace/CloSe/data/close_image_scan/10029_3362.png',
  '/workspace/CloSe/data/close_image